In [2]:
# Cell 1: Environment Initialization
import os
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from typing import Dict, Any, Tuple

In [3]:
# Cell 2: Ingestion & Sanity Inspection
DATASET_PATH = 'data/SYNTHETIC_DATASET_100_CLEANED.csv'

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}. Please verify your file path.")

df_raw = pd.read_csv(DATASET_PATH)
print(f"Dataset Loaded Successfully: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns.")

# Display key parameter distributions
display(df_raw[['id', 'mn', 'z1', 'z2', 'beta', 'b', 'T1', 'n1', 'Mtrue', 'TEtrue', 'Sound']].head(5))


Dataset Loaded Successfully: 100 rows x 51 columns.


,id,mn,z1,z2,beta,b,T1,n1,Mtrue,TEtrue,Sound
0,1,2.00,18,118,22,32,43.5,12200,31.7,0.93,51.9
1,2,2.25,18,120,18,30,65.6,9550,50.0,1.20,54.3
2,3,1.75,19,154,21,16,20.0,14600,23.1,1.39,62.0
3,4,1.50,18,166,23,16,9.7,15150,20.0,1.92,56.2
4,5,2.50,22,104,17,36,142.9,14800,65.5,1.39,57.0


In [4]:
# Cell 3: Enterprise Validation Schema
class HelicalGearInputSchema(BaseModel):
    """
    Strict validation schema enforcing physical and manufacturing limits
    per ISO 54, ISO 1328, and ISO 6336 standards.
    """
    # Macro-Geometry
    mn: float = Field(..., ge=1.0, le=10.0, description="Normal module [mm]")
    z1: int = Field(..., ge=10, le=100, description="Number of teeth - Pinion")
    z2: int = Field(..., ge=10, le=250, description="Number of teeth - Gear/Wheel")
    beta: float = Field(..., ge=10.0, le=45.0, description="Helix angle [deg]")
    alpha: float = Field(default=20.0, ge=14.5, le=25.0, description="Normal pressure angle [deg]")
    b: float = Field(..., ge=10.0, le=150.0, description="Face width [mm]")
    x1: float = Field(default=0.0, ge=-0.5, le=0.8, description="Pinion profile shift coefficient")
    x2: float = Field(default=0.0, ge=-0.5, le=0.8, description="Gear profile shift coefficient")
    
    # Operating Loads & Kinematics
    T1: float = Field(..., gt=0.0, description="Input torque at pinion [Nm]")
    n1: float = Field(..., gt=0.0, description="Input speed at pinion [rpm]")
    i: float = Field(..., gt=0.0, description="Gear ratio (z2/z1)")
    P: float = Field(default=0.0, description="Nominal input power [kW]")
    K_A: float = Field(default=1.15, ge=1.0, le=2.2, description="Application shock factor")
    
    # Micro-Geometry & Quality Grades
    Calpha: float = Field(default=10.0, ge=0.0, le=40.0, description="Tip relief [um]")
    Cbeta: float = Field(default=8.0, ge=0.0, le=30.0, description="Lead crowning [um]")
    Q: int = Field(default=6, ge=3, le=12, description="ISO 1328 Quality Grade")
    
    # Material & Thermal Defaults
    HRC: float = Field(default=60.0, ge=50.0, le=68.0, description="Surface Hardness [HRC]")
    HB: float = Field(default=315.0, description="Core Hardness [HB]")
    CHD: float = Field(default=0.62, ge=0.2, le=3.0, description="Case depth [mm]")
    sigma_H0: float = Field(default=1480.0, description="Allowable contact stress [MPa]")
    sigma_F0: float = Field(default=500.0, description="Allowable bending stress [MPa]")
    Ra: float = Field(default=0.8, ge=0.1, le=3.2, description="Flank roughness [um]")
    Toil: float = Field(default=70.0, description="Bulk oil temperature [°C]")
    VG: int = Field(default=32, description="Lubricant ISO VG viscosity")

print("Validation schema compiled successfully with engineering bounds.")


Validation schema compiled successfully with engineering bounds.


In [5]:
# Cell 4: ISO Mechanics Pre-Processor
def calculate_deterministic_gear_physics(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """
    Computes standard ISO 6336 / ISO 21771 derived physical properties:
      - Transverse pressure angle alpha_t
      - Pitch diameters d1, d2
      - Center distance a
      - Tangential tooth force Ft
      - Specific line load wt
      - Pitch-line velocity v
      - Fundamental mesh frequency fm
      - Contact ratios: eps_alpha, eps_beta, eps_gamma
    """
    mn = float(inputs['mn'])
    z1 = int(inputs['z1'])
    z2 = int(inputs['z2'])
    beta_deg = float(inputs['beta'])
    alpha_deg = float(inputs.get('alpha', 20.0))
    b = float(inputs['b'])
    T1 = float(inputs['T1'])
    n1 = float(inputs['n1'])
    
    # Radians conversion
    beta_rad = np.radians(beta_deg)
    alpha_n_rad = np.radians(alpha_deg)
    
    # Transverse Pressure Angle: tan(alpha_t) = tan(alpha_n) / cos(beta)
    alpha_t_rad = np.arctan(np.tan(alpha_n_rad) / np.cos(beta_rad))
    alpha_t_deg = np.degrees(alpha_t_rad)
    
    # Reference Pitch Diameters [mm]
    d1 = (mn * z1) / np.cos(beta_rad)
    d2 = (mn * z2) / np.cos(beta_rad)
    
    # Operating Center Distance [mm]
    a = (d1 + d2) / 2.0
    
    # Linear Pitch-line Velocity [m/s]
    v = (np.pi * d1 * n1) / 60000.0
    
    # Tangential Mesh Force [N]
    Ft = (2000.0 * T1) / d1
    
    # Base Helix Angle: cos(beta_b) = cos(beta) * cos(alpha_t) / cos(alpha_n)
    cos_beta_b = np.cos(beta_rad) * np.cos(alpha_t_rad) / np.cos(alpha_n_rad)
    
    # Specific Line Load wt [N/mm]
    wt = Ft / (b * cos_beta_b)
    
    # Fundamental Tooth Mesh Frequency [Hz]
    fm = (n1 * z1) / 60.0
    
    # Face Overlap Ratio (eps_beta)
    eps_beta = (b * np.sin(beta_rad)) / (np.pi * mn)
    
    # Transverse Profile Contact Ratio (eps_alpha standard approximation)
    eps_alpha = 1.88 - 3.2 * (1.0 / z1 + 1.0 / z2)
    
    # Total Contact Ratio (eps_gamma)
    eps_gamma = eps_alpha + eps_beta
    
    # Power calculation if not provided: P = (T1 * n1) / 9550 [kW]
    P = float(inputs.get('P', 0.0))
    if P <= 0.0:
        P = (T1 * n1) / 9550.0

    results = dict(inputs)
    results.update({
        'alpha_t_deg': float(round(alpha_t_deg, 3)),
        'd1_mm': float(round(d1, 4)),
        'd2_mm': float(round(d2, 4)),
        'a': float(round(a, 3)),
        'v_mps': float(round(v, 3)),
        'Ft_N': float(round(Ft, 3)),
        'wt_N_per_mm': float(round(wt, 3)),
        'fm_Hz': float(round(fm, 2)),
        'eps_alpha': float(round(max(1.1, eps_alpha), 3)),
        'eps_beta': float(round(max(0.5, eps_beta), 3)),
        'eps_gamma': float(round(max(1.6, eps_gamma), 3)),
        'P': float(round(P, 2))
    })
    return results

print("Deterministic ISO physics pre-processor ready.")


Deterministic ISO physics pre-processor ready.


In [6]:
# Cell 5: Automated Verification Test
benchmark_case_1 = {
    'mn': 2.0,
    'z1': 18,
    'z2': 118,
    'beta': 22.0,
    'alpha': 20.0,
    'b': 32.0,
    'T1': 43.5,
    'n1': 12200.0,
    'i': 6.56,
    'K_A': 1.13,
    'Calpha': 10.0,
    'Cbeta': 8.0,
    'Q': 5
}

# 1. Validate Schema
validated_obj = HelicalGearInputSchema(**benchmark_case_1)
print("Schema Check: Passed!")

# 2. Calculate Derived Physics
derived_metrics = calculate_deterministic_gear_physics(validated_obj.model_dump())

# 3. Compare with Reference Targets
expected_d1 = 38.83
expected_Ft = 2240.69
expected_fm = 3660.0

print("\n--- STAGE 1 VERIFICATION REPORT ---")
print(f"Pinion Pitch Diameter (d1):   {derived_metrics['d1_mm']:.2f} mm   (Target: {expected_d1:.2f} mm) -> Error: {abs(derived_metrics['d1_mm'] - expected_d1):.3f} mm")
print(f"Center Distance (a):          {derived_metrics['a']:.2f} mm   (Dataset a: 146.7 mm)")
print(f"Tangential Force (Ft):        {derived_metrics['Ft_N']:.2f} N    (Target: {expected_Ft:.2f} N) -> Error: {abs(derived_metrics['Ft_N'] - expected_Ft):.3f} N")
print(f"Specific Tooth Load (wt):     {derived_metrics['wt_N_per_mm']:.2f} N/mm")
print(f"Mesh Frequency (fm):          {derived_metrics['fm_Hz']:.1f} Hz   (Target: {expected_fm:.1f} Hz) -> Error: {abs(derived_metrics['fm_Hz'] - expected_fm):.3f} Hz")
print(f"Pitch-line Velocity (v):      {derived_metrics['v_mps']:.2f} m/s")
print(f"Overlap Ratio (eps_beta):     {derived_metrics['eps_beta']:.2f}")
print(f"Total Contact Ratio (eps_γ):  {derived_metrics['eps_gamma']:.2f}")

assert abs(derived_metrics['d1_mm'] - expected_d1) < 0.1, "d1 validation failed!"
assert abs(derived_metrics['Ft_N'] - expected_Ft) < 1.0, "Ft validation failed!"
assert abs(derived_metrics['fm_Hz'] - expected_fm) < 1.0, "fm validation failed!"
print("\nVERDICT: STAGE 1 PASSED WITH ZERO PHYSICAL DRIFT. READY FOR STAGE 2.")

Schema Check: Passed!

--- STAGE 1 VERIFICATION REPORT ---
Pinion Pitch Diameter (d1):   38.83 mm   (Target: 38.83 mm) -> Error: 0.003 mm
Center Distance (a):          146.68 mm   (Dataset a: 146.7 mm)
Tangential Force (Ft):        2240.69 N    (Target: 2240.69 N) -> Error: 0.004 N
Specific Tooth Load (wt):     76.24 N/mm
Mesh Frequency (fm):          3660.0 Hz   (Target: 3660.0 Hz) -> Error: 0.000 Hz
Pitch-line Velocity (v):      24.80 m/s
Overlap Ratio (eps_beta):     1.91
Total Contact Ratio (eps_γ):  3.58

VERDICT: STAGE 1 PASSED WITH ZERO PHYSICAL DRIFT. READY FOR STAGE 2.
